# 04 Time travel: then vs now

`python -m etl.time_travel` runs `etl/sql/80_time_travel_then_and_now.sql`. Versions of `gold/fact_edit` are found by their commit metadata `as_of`, never by timestamp. Here the same query is repeated with delta-rs, to show that both engines read the same versions.

In [1]:
import json, os, sys
from pathlib import Path
REPO = Path.cwd().resolve().parents[1] if Path.cwd().name == 'final' else Path.cwd()
sys.path.insert(0, str(REPO))
os.environ.setdefault('TRUTHLAG_HW2_DIR', str(REPO / 'ec-hw2'))
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
from deltalake import DeltaTable
LAKE = REPO / 'data' / 'lake'
RUNS = LAKE / '_runs'
def run(name): return json.loads((RUNS / name).read_text())
def table(name, version=None, columns=None):
    return DeltaTable(str(LAKE / name), version=version).to_pandas(columns=columns)
def history(name, n=10):
    h = DeltaTable(str(LAKE / name)).history(n)
    return pd.DataFrame([{k: e.get(k) for k in ('version', 'operation', 'step', 'as_of', 'snapshot', 'rows')} for e in h])
tt = run('time_travel.json')
print(tt['fact_edit_versions'])

{'early': 1, 'parity': 2, 'ext': 3}


## Snapshot axis: fact_revert_label as loaded from History snapshot 2026-09 and 2026-10

`etl/sql/80_time_travel_then_and_now.sql`: reverted history-cohort revisions as known at each snapshot's load (VERSION AS OF) and now.

In [2]:
print(tt['snapshot_axis']['fact_revert_label_versions'])
pd.DataFrame(tt['snapshot_axis']['then_and_now'])

{'2026-09': 1, '2026-10': 2}


,snapshot,label_version,reverted_as_known_then,reverted_now
0,2026-09,1,19858,20925
1,2026-10,2,20925,20925


## Stream axis: fact_edit at the three build as_of values (`etl/sql/81_time_travel_as_of.sql`)

In [3]:
pd.DataFrame(tt['then_and_now'])

,known_at,fact_edit_version,live_revisions,reverted_as_known,pct_reverted,newest_known_dt
0,then: as_of 2026-10-24T00:00:00.000000Z,1,279466,17040,6.097,2026-10-23 23:59:03.674000
1,parity: as_of 2026-10-27T20:59:59.999999Z,2,279466,17818,6.376,2026-10-26 18:30:28.229000
2,now: as_of 2026-11-13T23:59:59.999999Z,3,279466,17818,6.376,2026-10-26 18:30:28.229000


## The same, read by delta-rs at each version

In [4]:
rows = []
for k, v in tt['fact_edit_versions'].items():
    f = table('gold/fact_edit', version=v, columns=['cohort_id', 'live_revert_flag'])
    f = f[f.cohort_id == 'live']
    rows.append({'build': k, 'version': v, 'live_revisions': len(f), 'reverted_as_known': int(f.live_revert_flag.fillna(False).sum())})
pd.DataFrame(rows)

/var/folders/24/0v4v689d3v58yz4_ys65jzj00000gn/T/ipykernel_81992/1065713887.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rows.append({'build': k, 'version': v, 'live_revisions': len(f), 'reverted_as_known': int(f.live_revert_flag.fillna(False).sum())})


,build,version,live_revisions,reverted_as_known
0,early,1,279466,17040
1,parity,2,279466,17818
2,ext,3,279466,17818


## Change Data Feed of fact_edit between the versions

In [5]:
pd.DataFrame(tt['change_feed'])

,_commit_version,_change_type,cohort_id,n,flag_true
0,2,update_postimage,live,778,778
1,2,update_preimage,live,778,0
2,3,insert,live_ext,2397738,112937


In [6]:
tt['early_to_parity_updates']

{'updated_rows': 778,
 'became_reverted': 778,
 'page_key_changed': 0,
 'median_known_after_s': 245324}

## Silver by version: live_revert_links at each build

In [7]:
pd.DataFrame(tt['silver_live_revert_links_by_version'])

,version,as_of,links,within_followup
0,1,2026-10-24T00:00:00.000000Z,17040,17040
1,2,2026-10-27T20:59:59.999999Z,18358,17818
2,3,2026-11-13T23:59:59.999999Z,19513,17818
